In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

In [ ]:
train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")
normal = train[train["label"] != 24].drop(columns="label").to_numpy(dtype="float32") / 255
x_train, x_val = train_test_split(normal, test_size=0.1, random_state=42)
x_test = test.drop(columns="label").to_numpy(dtype="float32") / 255
y_test = test["label"].to_numpy() == 24

In [ ]:
model = Sequential([Input((784,)),
    Dense(128, activation="relu"),
    Dense(16, activation="relu"),
    Dense(128, activation="relu"),
    Dense(784, activation="sigmoid")])
model.compile(optimizer="adam", loss="mse")

In [ ]:
model.fit(x_train, x_train, epochs=30, batch_size=128, validation_data=(x_val, x_val))

In [ ]:
normal_error = np.mean((x_val - model.predict(x_val, verbose=0)) ** 2, axis=1)
threshold = np.percentile(normal_error, 95)

In [ ]:
test_error = np.mean((x_test - model.predict(x_test, verbose=0)) ** 2, axis=1)
anomalies = test_error > threshold
print("Test accuracy (%):", round(100 * accuracy_score(y_test, anomalies), 2))
print("Detection AUC:", roc_auc_score(y_test, test_error))